This page is dedicated to utilize BlackJAX package to reproduce the Numerical Experiment from Section 4 of Margossian et al. and run independent tests for improved workflow.

**To make it run on GPUs**

In [ ]:
# Remove the stable TFP package
!pip uninstall -yq tensorflow-probability
!pip uninstall -yq jax jaxlib jax-cuda12-plugin jax-cuda12-pjrt
# Install nightly TFP for JAX
!pip install -Uq tfp-nightly[jax]
# Install Inference Gym
!pip install inference-gym
# need to keep it cuda13 cuz blackjax drop cuda12 support
!pip install -Uq "jax[cuda13]" blackjax optax arviz-base arviz-stats

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.3/3.3 MB 120.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 87.3/87.3 MB 31.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 16.7/16.7 MB 136.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 7.0/7.0 MB 142.4 MB/s eta 0:00:00
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
dopamine-rl 4.1.2 requires tensorflow-probability>=0.13.0, which is not installed.
numba 0.60.0 requires numpy<2.1,>=1.22, but you have numpy 2.5.2 which is incompatible.
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 390.9/390.9 kB 30.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 4.9/4.9 MB 136.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.4/1.4 MB 89.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 183.8/183.8 kB 20.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━

In [ ]:
# run those checks if package compatibility is in trouble

# import jax
# import tensorflow_probability as tfp
# import jaxlib

# print("jaxlib:", jaxlib.__version__)
# print("TFP:", tfp.__version__)

# !pip show jax
# !pip show jaxlib
# !pip show blackjax

# import jax.numpy as jnp

# import blackjax

# import tensorflow_probability.substrates.jax as tfp
# import inference_gym.using_jax as gym

# print("JAX:", jax.__version__)
# print("BlackJAX:", blackjax.__version__)
# print("TFP:", tfp.__version__)
# print("ArviZ:", avs.__version__)
# print("Inference Gym imported successfully!")

jax: 0.11.0
jaxlib: 0.11.0
Name: jax
Version: 0.11.0
Summary: Differentiate, compile, and transform Numpy code.
Home-page: https://github.com/jax-ml/jax
Author: JAX team
Author-email: jax-dev@google.com
License: Apache-2.0
Location: /usr/local/lib/python3.12/dist-packages
Requires: jaxlib, ml_dtypes, numpy, opt_einsum, scipy
Required-by: blackjax, dopamine_rl, flax, optax, orbax-checkpoint
Name: jaxlib
Version: 0.11.0
Summary: XLA library for JAX
Home-page: https://github.com/jax-ml/jax
Author: JAX team
Author-email: jax-dev@google.com
License: Apache-2.0
Location: /usr/local/lib/python3.12/dist-packages
Requires: ml_dtypes, numpy, scipy
Required-by: blackjax, dopamine_rl, jax, optax
Name: blackjax
Version: 1.6.2
Summary: Flexible and fast sampling in Python
Home-page: https://github.com/blackjax-devs/blackjax
Author: 
Author-email: The Blackjax team <remi@thetypicalset.com>
License: Apache License 2.0
Location: /usr/local/lib/python3.12/dist-packages
Requires: jax, jaxlib, numpy, opta

**Necessary Packages and Other Settings**


In [ ]:
# GPU set up to accelerate performance
import os
# in case jax eats up my GPU RAM
os.environ["XLA_PYTHON_CLIENT_PREALLOCATE"] = "false"
os.environ['XLA_FLAGS'] = (
    '--xla_gpu_triton_gemm_any=True '
    '--xla_gpu_enable_latency_hiding_scheduler=true '
)

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

import jax
import jax.numpy as jnp
from jax import random, jit, vmap, lax
import tensorflow_probability.substrates.jax as tfp
tfd = tfp.distributions
import inference_gym.using_jax as gym
import jaxlib
import blackjax

import optax

import arviz as az
import arviz_stats as avs

import warnings
warnings.filterwarnings('ignore')

import psutil

import gc

from google.colab import drive
from matplotlib.lines import Line2D

process = psutil.Process(os.getpid())

def mem(msg):
    print(f"{msg}: {process.memory_info().rss / 1024**2:.1f} MB")

# verification to make sure this is on a GPU
print(jax.devices())
print(jax.default_backend())

[CudaDevice(id=0)]
gpu


In [ ]:
drive.mount('/content/drive')
utility_link = '/content/drive/MyDrive/Colab Notebooks/2026_Summer_MCMC/BJAX_files/BlackJAXUtil.py'
with open(utility_link) as f: exec(f.read())

Mounted at /content/drive


In [ ]:
max_warmup = 1000
warmup_window = 100

window_array = np.append(np.repeat(10, 10),
                      np.repeat(warmup_window, max_warmup // warmup_window - 1))

warmup_length = np.repeat(10, len(window_array))
for i in range(len(warmup_length) - 1):
    warmup_length[i + 1] = warmup_length[i] + window_array[i + 1]

# Transition kernel for short regime
repitition = 10
num_chains_short = 2048
num_super_chains = 16

In [ ]:
# quantiles for chi squared with df = 1
chi_up = 3.841459 # 95th quantile for chi squared with df = 1
chi_lo = 0.00393214  # 05th quantile for chi squared with df = 1
tau = 1e-4
M = num_chains_short // num_super_chains
nRhat_lower = np.sqrt(1 + 1 / M + tau)
eps_lower = nRhat_lower - 1
bound = [chi_lo / num_chains_short, chi_up / num_chains_short]
threshold = eps_lower

**Rosenbrock Example**

In [ ]:
target = gym.targets.VectorModel(
    gym.targets.Banana(),
    flatten_sample_transformations=True
)

num_dimensions = target.event_shape[0]

# print("Target:", type(target))
# print("Dimensions:", num_dimensions)
# print("Event shape:", target.event_shape)
# Get some estimates of the mean and variance.
try:
  mean_est = target.sample_transformations['identity'].ground_truth_mean
except:
  print('no ground truth mean')
  mean_est = (result.all_states[num_warmup:, :]).mean(0).mean(0)
try:
  var_est = target.sample_transformations['identity'].ground_truth_standard_deviation**2
except:
  print('no ground truth std dev')
  var_est = ((result.all_states[num_warmup:, :]**2).mean(0).mean(0) -
             mean_est**2)
mean_benchmark = mean_est
var_benchmark = var_est

In [ ]:
def logdensity(x):
    y = target.default_event_space_bijector(x)
    fldj = target.default_event_space_bijector.forward_log_det_jacobian(x)
    return target.unnormalized_log_prob(y) + fldj
offset = 2.0
initial_step_size = 1.
def initialize(shape, key):
    return (10 * random.normal(key, shape+(num_dimensions,))+ offset)

In [ ]:
num_chains = num_chains_short      # 2048
num_super_chains = num_super_chains # 16
num_sub_chains = num_chains // num_super_chains  # 128

In [ ]:
#simulation part:
Rosen_MSE_c_list = []
Rosen_MSE_n_list = []
Rosen_RHat_c_list = []
Rosen_RHat_n_list = []
Rosen_state_list_c = []
Rosen_state_list_n = []

In [ ]:
base_key = random.PRNGKey(0)
keys = random.split(base_key, repitition)
for length in warmup_length:
  mem(f"Simulation Start")
  simulation(length,num_chains_short, num_super_chains,
             False,initialize, keys,
             logdensity,initial_step_size,
             repitition, Rosen_RHat_c_list,Rosen_MSE_c_list,
             mean_benchmark,var_benchmark)
  simulation(length,num_chains_short, num_super_chains,
             True,initialize, keys,
             logdensity,initial_step_size,
             repitition, Rosen_RHat_n_list,Rosen_MSE_n_list,
             mean_benchmark,var_benchmark)

Simulation Start: 1855.7 MB
Constrained initialization. Warmup Length: 10; mean of MSE is: 0.04272257164120674
Naive initialization. Warmup Length: 10; mean of MSE is: 0.013931222259998322
Simulation Start: 2282.5 MB
Constrained initialization. Warmup Length: 20; mean of MSE is: 0.030453717336058617
Naive initialization. Warmup Length: 20; mean of MSE is: 0.011684535071253777
Simulation Start: 2305.1 MB
Constrained initialization. Warmup Length: 30; mean of MSE is: 0.022949405014514923
Naive initialization. Warmup Length: 30; mean of MSE is: 0.01007282454520464
Simulation Start: 2313.0 MB
Constrained initialization. Warmup Length: 40; mean of MSE is: 0.018924809992313385
Naive initialization. Warmup Length: 40; mean of MSE is: 0.00891608651727438
Simulation Start: 2328.4 MB
Constrained initialization. Warmup Length: 50; mean of MSE is: 0.01634620688855648
Naive initialization. Warmup Length: 50; mean of MSE is: 0.007183185312896967
Simulation Start: 2334.3 MB
Constrained initialization

In [ ]:
RHat_c_df = pd.DataFrame(Rosen_RHat_c_list)
RHat_n_df = pd.DataFrame(Rosen_RHat_n_list)
MSE_c_df = pd.DataFrame(Rosen_MSE_c_list)
MSE_n_df = pd.DataFrame(Rosen_MSE_n_list)

In [ ]:
MSE_c_df.to_pickle(
    "/content/drive/MyDrive/Colab Notebooks/2026_Summer_MCMC/BlackJAX_pkl_files/Rosen_MSE_constrained.pkl"
)

MSE_n_df.to_pickle(
    "/content/drive/MyDrive/Colab Notebooks/2026_Summer_MCMC/BlackJAX_pkl_files/Rosen_MSE_naive.pkl"
)

RHat_c_df.to_pickle(
    "/content/drive/MyDrive/Colab Notebooks/2026_Summer_MCMC/BlackJAX_pkl_files/Rosen_R_Hat_constrained.pkl"
)

RHat_n_df.to_pickle(
    "/content/drive/MyDrive/Colab Notebooks/2026_Summer_MCMC/BlackJAX_pkl_files/Rosen_R_Hat_naive.pkl"
)

**Bimodal Example**

In [ ]:
num_dimensions = 100
offset = 0.0
init_step_size = 1.0

def target_log_prob_fn(x):
    logp1 = (
        jnp.log(0.3)
        - 0.5 * jnp.sum((x + 5.0) ** 2)
        - 0.5 * num_dimensions * jnp.log(2 * jnp.pi)
    )

    logp2 = (
        jnp.log(0.7)
        - 0.5 * jnp.sum((x - 5.0) ** 2)
        - 0.5 * num_dimensions * jnp.log(2 * jnp.pi)
    )

    return jax.scipy.special.logsumexp(
        jnp.array([logp1, logp2])
    )

def initialize (shape, key):
  return 10 * random.normal(key, shape + (num_dimensions,)) + offset

mean_est = jnp.repeat(2, num_dimensions)
var_est = jnp.repeat(22, num_dimensions)
mean_benchmark = mean_est
var_benchmark = var_est

In [ ]:
num_chains = num_chains_short      # 2048
num_super_chains = num_super_chains # 16
num_sub_chains = num_chains // num_super_chains  # 128

In [ ]:
#simulation part:
Bim_MSE_c_list = []
Bim_MSE_n_list = []
Bim_RHat_c_list = []
Bim_RHat_n_list = []
Bim_state_list_c = []
Bim_state_list_n = []

In [ ]:
base_key = random.PRNGKey(0)
keys = random.split(base_key, repitition)
for length in warmup_length:
  mem(f"Simulation Start")
  simulation(length,num_chains_short, num_super_chains,
             False,initialize, keys,
             target_log_prob_fn,init_step_size,
             repitition, Bim_RHat_c_list,Bim_MSE_c_list,
             mean_benchmark,var_benchmark)
  simulation(length,num_chains_short, num_super_chains,
             True,initialize, keys,
             target_log_prob_fn,init_step_size,
             repitition, Bim_RHat_n_list,Bim_MSE_n_list,
             mean_benchmark,var_benchmark)

Simulation Start: 2404.9 MB
Constrained initialization. Warmup Length: 10; mean of MSE is: 0.19671212136745453
Naive initialization. Warmup Length: 10; mean of MSE is: 0.18647286295890808
Simulation Start: 2429.4 MB
Constrained initialization. Warmup Length: 20; mean of MSE is: 0.1969003975391388
Naive initialization. Warmup Length: 20; mean of MSE is: 0.18654975295066833
Simulation Start: 2438.7 MB
Constrained initialization. Warmup Length: 30; mean of MSE is: 0.19683155417442322
Naive initialization. Warmup Length: 30; mean of MSE is: 0.18667301535606384
Simulation Start: 2447.6 MB
Constrained initialization. Warmup Length: 40; mean of MSE is: 0.1967579424381256
Naive initialization. Warmup Length: 40; mean of MSE is: 0.1864558458328247
Simulation Start: 2464.2 MB
Constrained initialization. Warmup Length: 50; mean of MSE is: 0.1970207393169403
Naive initialization. Warmup Length: 50; mean of MSE is: 0.186669260263443
Simulation Start: 2488.4 MB
Constrained initialization. Warmup Len

In [ ]:
RHat_c_df = pd.DataFrame(Bim_RHat_c_list)
RHat_n_df = pd.DataFrame(Bim_RHat_n_list)
MSE_c_df = pd.DataFrame(Bim_MSE_c_list)
MSE_n_df = pd.DataFrame(Bim_MSE_n_list)

In [ ]:
MSE_c_df.to_pickle(
    "/content/drive/MyDrive/Colab Notebooks/2026_Summer_MCMC/BlackJAX_pkl_files/Bim_MSE_constrained.pkl"
)

MSE_n_df.to_pickle(
    "/content/drive/MyDrive/Colab Notebooks/2026_Summer_MCMC/BlackJAX_pkl_files/Bim_MSE_naive.pkl"
)

RHat_c_df.to_pickle(
    "/content/drive/MyDrive/Colab Notebooks/2026_Summer_MCMC/BlackJAX_pkl_files/Bim_R_Hat_constrained.pkl"
)

RHat_n_df.to_pickle(
    "/content/drive/MyDrive/Colab Notebooks/2026_Summer_MCMC/BlackJAX_pkl_files/Bim_R_Hat_naive.pkl"
)

**Eight Schools Example**

In [ ]:
# NOTE: inference gym stores the centered parameterization
target_raw = gym.targets.EightSchools()  # store raw to examine doc.
target = gym.targets.VectorModel(target_raw,
                                  flatten_sample_transformations = True)
num_dimensions = target.event_shape[0]
init_step_size = 1.

def initialize (shape, key):
    prior_scale = jnp.append(jnp.array([10., 1.]), jnp.repeat(1., 8))
    prior_offset = jnp.append(jnp.array([0., 5.]), jnp.repeat(0., 8))
    return prior_scale * random.normal(key, shape + (num_dimensions,)) + prior_offset

num_schools = 8
y = np.array([28, 8, -3, 7, -1, 1, 18, 12], dtype = np.float32)
sigma = np.array([15, 10, 16, 11, 9, 11, 10, 18], dtype = np.float32)

# NOTE: the reinterpreted batch dimension specifies the dimension of
# each indepdent variable, here the school.
model = tfd.JointDistributionSequential([
    tfd.Normal(loc = 0., scale = 10., name = "mu"),
    tfd.Normal(loc = 5., scale = 1., name = "log_tau"),
    tfd.Independent(tfd.Normal(loc = jnp.zeros(num_schools),
                               scale = jnp.ones(num_schools),
                               name = "eta"),
                    reinterpreted_batch_ndims = 1),
    lambda eta, log_tau, mu: (
        tfd.Independent(tfd.Normal(loc = (mu[..., jnp.newaxis] +
                                        jnp.exp(log_tau[..., jnp.newaxis]) *
                                        eta),
                                   scale = sigma),
                        name = "y",
                        reinterpreted_batch_ndims = 1))
  ])

# minor change from the TFP code
# def target_log_prob_fn(x):
#   mu = x[:, 0]
#   log_tau = x[:, 1]
#   eta = x[:, 2:10]
#   return model.log_prob((mu, log_tau, eta, y))
def target_log_prob_fn(x):
  mu = x[0]
  log_tau = x[1]
  eta = x[2:10]
  return model.log_prob((mu, log_tau, eta, y))

In [ ]:
# Use results from running 128 chains with 1000 + 5000 iterations each,
# for non-centered parameterization.
mean_est = np.array([5.8006573 ,  2.4502006 ,  0.6532423 ,  0.09639207,
             -0.23725411,  0.04723661, -0.33556408, -0.19666635,
              0.5390533 ,  0.14633301])

var_est = np.array([29.60382   ,  0.26338503,  0.6383733 ,  0.4928926 ,
              0.65307987,  0.52441144,  0.46658015,  0.5248887 ,
              0.49544162,  0.690975])
mean_benchmark = mean_est
var_benchmark = var_est

In [ ]:
#simulation part:
base_key = random.PRNGKey(0)
keys = random.split(base_key, repitition)

School_MSE_c_list = []
School_MSE_n_list = []
School_RHat_c_list = []
School_RHat_n_list = []
School_state_list_c = []
School_state_list_n = []

In [ ]:
for length in warmup_length:
  mem(f"Simulation Start")
  simulation(length,num_chains_short, num_super_chains,
             False,initialize, keys,
             target_log_prob_fn,init_step_size,
             repitition, School_RHat_c_list,School_MSE_c_list,
             mean_benchmark,var_benchmark)
  simulation(length,num_chains_short, num_super_chains,
             True,initialize, keys,
             target_log_prob_fn,init_step_size,
             repitition, School_RHat_n_list,School_MSE_n_list,
             mean_benchmark,var_benchmark)

Simulation Start: 1876.5 MB
Constrained initialization. Warmup Length: 10; mean of MSE is: 6.584364414215088
Naive initialization. Warmup Length: 10; mean of MSE is: 3.8581595420837402
Simulation Start: 2352.3 MB
Constrained initialization. Warmup Length: 20; mean of MSE is: 0.2077944278717041
Naive initialization. Warmup Length: 20; mean of MSE is: 0.12716639041900635
Simulation Start: 2355.8 MB
Constrained initialization. Warmup Length: 30; mean of MSE is: 0.11714692413806915
Naive initialization. Warmup Length: 30; mean of MSE is: 0.10511034727096558
Simulation Start: 2357.2 MB
Constrained initialization. Warmup Length: 40; mean of MSE is: 0.10378307104110718
Naive initialization. Warmup Length: 40; mean of MSE is: 0.09946390986442566
Simulation Start: 2357.9 MB
Constrained initialization. Warmup Length: 50; mean of MSE is: 0.08759568631649017
Naive initialization. Warmup Length: 50; mean of MSE is: 0.08984960615634918
Simulation Start: 2358.2 MB
Constrained initialization. Warmup L

In [ ]:
RHat_c_df = pd.DataFrame(School_RHat_c_list)
RHat_n_df = pd.DataFrame(School_RHat_n_list)
MSE_c_df = pd.DataFrame(School_MSE_c_list)
MSE_n_df = pd.DataFrame(School_MSE_n_list)

In [ ]:
MSE_c_df.to_pickle(
    "/content/drive/MyDrive/Colab Notebooks/2026_Summer_MCMC/BlackJAX_pkl_files/School_MSE_constrained.pkl"
)

MSE_n_df.to_pickle(
    "/content/drive/MyDrive/Colab Notebooks/2026_Summer_MCMC/BlackJAX_pkl_files/School_MSE_naive.pkl"
)

RHat_c_df.to_pickle(
    "/content/drive/MyDrive/Colab Notebooks/2026_Summer_MCMC/BlackJAX_pkl_files/School_R_Hat_constrained.pkl"
)

RHat_n_df.to_pickle(
    "/content/drive/MyDrive/Colab Notebooks/2026_Summer_MCMC/BlackJAX_pkl_files/School_R_Hat_naive.pkl"
)

**Item Response Theory Example**

In [ ]:
target = gym.targets.VectorModel(gym.targets.SyntheticItemResponseTheory(),
                                 flatten_sample_transformations=True)
num_dimensions = target.event_shape[0]
init_step_size = 1.

def target_log_prob_fn(x):
  """Unnormalized, unconstrained target density.

  This is a thin wrapper that applies the default bijectors so that we can
  ignore any constraints.
  """
  y = target.default_event_space_bijector(x)
  fldj = target.default_event_space_bijector.forward_log_det_jacobian(x)
  return target.unnormalized_log_prob(y) + fldj

offset = 0
def initialize (shape, key):
  return 10 * random.normal(key, shape + (num_dimensions,)) + offset

In [ ]:
# Get some estimates of the mean and variance.
try:
  mean_est = target.sample_transformations['identity'].ground_truth_mean
except:
  print('no ground truth mean')
  mean_est = (result.all_states[num_warmup:, :]).mean(0).mean(0)
try:
  var_est = target.sample_transformations['identity'].ground_truth_standard_deviation**2
except:
  print('no ground truth std dev')
  var_est = ((result.all_states[num_warmup:, :]**2).mean(0).mean(0) -
             mean_est**2)

mean_benchmark = mean_est
var_benchmark = var_est

In [ ]:
IRT_MSE_c_list = []
IRT_MSE_n_list = []
IRT_RHat_c_list = []
IRT_RHat_n_list = []
IRT_state_list_c = []
IRT_state_list_n = []

base_key = random.PRNGKey(0)
keys = random.split(base_key, repitition)

In [ ]:
for length in warmup_length:
  mem(f"Simulation Start")
  simulation(length,num_chains_short, num_super_chains,
             False,initialize, keys,
             target_log_prob_fn,init_step_size,
             repitition,IRT_RHat_c_list,IRT_MSE_c_list,
             mean_benchmark,var_benchmark)
  simulation(length,num_chains_short, num_super_chains,
             True,initialize, keys,
             target_log_prob_fn,init_step_size,
             repitition, IRT_RHat_n_list,IRT_MSE_n_list,
             mean_benchmark,var_benchmark)

Simulation Start: 1871.1 MB
Constrained initialization. Warmup Length: 10; mean of MSE is: 12.777505874633789
Naive initialization. Warmup Length: 10; mean of MSE is: 17.213642120361328
Simulation Start: 2680.0 MB
Constrained initialization. Warmup Length: 20; mean of MSE is: 0.10465814918279648
Naive initialization. Warmup Length: 20; mean of MSE is: 0.09344545751810074
Simulation Start: 2770.2 MB
Constrained initialization. Warmup Length: 30; mean of MSE is: 0.0031073063146322966
Naive initialization. Warmup Length: 30; mean of MSE is: 0.0005827909335494041
Simulation Start: 2855.3 MB
Constrained initialization. Warmup Length: 40; mean of MSE is: 0.001718431361950934
Naive initialization. Warmup Length: 40; mean of MSE is: 0.0004552365280687809
Simulation Start: 2936.7 MB
Constrained initialization. Warmup Length: 50; mean of MSE is: 0.0008129449561238289
Naive initialization. Warmup Length: 50; mean of MSE is: 0.00048781270743347704
Simulation Start: 3020.7 MB
Constrained initializa

In [ ]:
RHat_c_df = pd.DataFrame(IRT_RHat_c_list)
RHat_n_df = pd.DataFrame(IRT_RHat_n_list)
MSE_c_df = pd.DataFrame(IRT_MSE_c_list)
MSE_n_df = pd.DataFrame(IRT_MSE_n_list)

In [ ]:
MSE_c_df.to_pickle(
    "/content/drive/MyDrive/Colab Notebooks/2026_Summer_MCMC/BlackJAX_pkl_files/IRT_MSE_constrained.pkl"
)

MSE_n_df.to_pickle(
    "/content/drive/MyDrive/Colab Notebooks/2026_Summer_MCMC/BlackJAX_pkl_files/IRT_MSE_naive.pkl"
)

RHat_c_df.to_pickle(
    "/content/drive/MyDrive/Colab Notebooks/2026_Summer_MCMC/BlackJAX_pkl_files/IRT_R_Hat_constrained.pkl"
)

RHat_n_df.to_pickle(
    "/content/drive/MyDrive/Colab Notebooks/2026_Summer_MCMC/BlackJAX_pkl_files/IRT_R_Hat_naive.pkl"
)